In [1]:
import numpy as np
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error

# ---- Grey Wolf Optimizer ----
class GreyWolfOptimizer:
    def __init__(self, obj_func, dim, lb, ub, search_agents=5, max_iter=10):
        self.obj_func = obj_func
        self.dim = dim
        self.lb = np.array(lb)
        self.ub = np.array(ub)
        self.search_agents = search_agents
        self.max_iter = max_iter

    def optimize(self):
        alpha, beta, delta = np.zeros(self.dim), np.zeros(self.dim), np.zeros(self.dim)
        alpha_score = beta_score = delta_score = float('inf')
        wolves = np.random.uniform(self.lb, self.ub, (self.search_agents, self.dim))

        for t in range(self.max_iter):
            a = 2 - t * (2 / self.max_iter)
            for i in range(self.search_agents):
                wolves[i] = np.clip(wolves[i], self.lb, self.ub)
                score = self.obj_func(wolves[i])
                if score < alpha_score:
                    alpha_score, alpha = score, wolves[i].copy()
                elif score < beta_score:
                    beta_score, beta = score, wolves[i].copy()
                elif score < delta_score:
                    delta_score, delta = score, wolves[i].copy()

            for i in range(self.search_agents):
                for j in range(self.dim):
                    r1, r2 = np.random.rand(), np.random.rand()
                    A1, C1 = 2*a*r1 - a, 2*r2
                    D_alpha = abs(C1*alpha[j] - wolves[i, j])
                    X1 = alpha[j] - A1*D_alpha

                    r1, r2 = np.random.rand(), np.random.rand()
                    A2, C2 = 2*a*r1 - a, 2*r2
                    D_beta = abs(C2*beta[j] - wolves[i, j])
                    X2 = beta[j] - A2*D_beta

                    r1, r2 = np.random.rand(), np.random.rand()
                    A3, C3 = 2*a*r1 - a, 2*r2
                    D_delta = abs(C3*delta[j] - wolves[i, j])
                    X3 = delta[j] - A3*D_delta

                    wolves[i, j] = (X1 + X2 + X3) / 3
        return alpha, alpha_score


In [2]:

# ---- Data Preparation ----
data = fetch_california_housing()
X_train, X_test, y_train, y_test = train_test_split(
    data.data, data.target, test_size=0.2, random_state=42
)

# ---- Objective Function ----
def rf_obj_func(params):
    n_estimators = int(params[0])
    max_depth = int(params[1])
    n_estimators = max(10, n_estimators)   # enforce minimums
    max_depth = max(1, max_depth)
    model = RandomForestRegressor(n_estimators=n_estimators, max_depth=max_depth, random_state=42)
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    return rmse

# ---- Run GWO ----
lb = [10, 1]   # lower bounds for n_estimators and max_depth
ub = [100, 20] # upper bounds
gwo = GreyWolfOptimizer(rf_obj_func, dim=2, lb=lb, ub=ub, search_agents=5, max_iter=10)
best_params, best_score = gwo.optimize()
print(f'Best parameters: n_estimators={int(best_params[0])}, max_depth={int(best_params[1])}, RMSE={best_score:.4f}')


Best parameters: n_estimators=97, max_depth=20, RMSE=0.5058
